In [1]:
import coiled

import fsspec
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import dask
import sparse
from dask.distributed import Client, LocalCluster
from dask.distributed import print

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=2,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [ ]:
client.restart() 

In [6]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 14,Total memory: 31.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45887,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 31.08 GiB
Comm: tcp://127.0.0.1:36893,Total threads: 2
Dashboard: http://127.0.0.1:33383/status,Memory: 4.44 GiB
Nanny: tcp://127.0.0.1:35503,


In [ ]:
local_client.shutdown()

In [ ]:
# cluster.adapt(minimum=25, maximum=100)

In [ ]:
# tcl_tiles = pd.read_json('s3://gfw-data-lake/umd_tree_cover_loss/v1.11/raster/epsg-4326/10/40000/year/gdal-geotiff/tiles.geojson')
# areas_tiles = pd.read_json('s3://gfw-data-lake/umd_area_2013/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/tiles.geojson')
# gadm_adm0_tiles = pd.read_json('s3://gfw-data-lake/gadm_administrative_boundaries/v4.1.64/raster/epsg-4326/10/40000/adm0/gdal-geotiff/tiles.geojson')
# gadm_adm1_tiles = pd.read_json('s3://gfw-data-lake/gadm_administrative_boundaries/v4.1.64/raster/epsg-4326/10/40000/adm1/gdal-geotiff/tiles.geojson')
# gadm_adm2_tiles = pd.read_json('s3://gfw-data-lake/gadm_administrative_boundaries/v4.1.64/raster/epsg-4326/10/40000/adm2/gdal-geotiff/tiles.geojson')
# drivers_tiles = pd.read_json('s3://gfw-data-lake/tsc_tree_cover_loss_drivers/v2023/raster/epsg-4326/10/40000/driver/gdal-geotiff/tiles.geojson')
# tcd_thresholds_tiles = pd.read_json('s3://gfw-data-lake/umd_tree_cover_density_2000/v1.8/raster/epsg-4326/10/40000/threshold/gdal-geotiff/tiles.geojson')

tcl_tile_test = pd.read_json('s3://gfw-files/dgibbs/flox_test/tiles.geojson')
print(tcl_tile_test)

In [ ]:
def get_uri(feature):
    raw = feature['properties']['name'].split('/')[2:]
    uri = '/'.join(['s3:/'] + raw)
    return uri

In [ ]:
# tcl_uris = tcl_tiles.features.apply(get_uri)
# areas_uris = areas_tiles.features.apply(get_uri)
# drivers_uris = drivers_tiles.features.apply(get_uri)
# gadm_adm0_uris = gadm_adm0_tiles.features.apply(get_uri)
# gadm_adm1_uris = gadm_adm1_tiles.features.apply(get_uri)
# gadm_adm2_uris = gadm_adm2_tiles.features.apply(get_uri)
# tcd_thresholds_uris = tcd_thresholds_tiles.features.apply(get_uri)
# print(tcl_uris)

tcl_tile = tcl_tile_test.features.apply(get_uri)
print(tcl_tile)
print(type(tcl_tile))
    

In [ ]:
import flox
flox.__version__

In [ ]:
areas_sub, drivers_aligned = xr.align(areas, drivers, join="inner")


In [ ]:
_, tcl_aligned = xr.align(areas_sub, tcl_year, join="left")
_, gadm_adm0_aligned = xr.align(areas_sub, gadm_adm0, join="left")
_, gadm_adm1_aligned = xr.align(areas_sub, gadm_adm1, join="left")
_, gadm_adm2_aligned = xr.align(areas_sub, gadm_adm2, join="left")
# _, drivers_aligned = xr.align(areas, drivers, join="left")
_, tcd_thresholds_aligned = xr.align(areas_sub, tcd_thresholds, join="left")


In [ ]:
drivers_aligned = drivers_aligned.astype(np.uint8)

In [ ]:
# tcl_year = xr.open_mfdataset(
#     tcl_uris.values.tolist(),
#     parallel=True,
#     chunks={'x': 10000, 'y':10000}
# ).squeeze().astype(np.uint8).persist()

# areas = xr.open_mfdataset(
#     areas_uris.values.tolist(),
#     parallel=True,
#     chunks={'x': 10000, 'y':10000}
# ).squeeze().persist()

tcl = xr.open_mfdataset(
    tcl_tile.values.tolist(),
    parallel=True,
    chunks={'x': 10000, 'y':10000}
).squeeze().persist()

tcl


In [ ]:
tcl_data = tcl_aligned.band_data
tcl_data.name = 'tcl_year'

gadm_adm0_data = gadm_adm0_aligned.band_data
gadm_adm0_data.name = 'gadm_adm0'

gadm_adm1_data = gadm_adm1_aligned.band_data
gadm_adm1_data.name = 'gadm_adm1'

gadm_adm2_data = gadm_adm2_aligned.band_data
gadm_adm2_data.name = 'gadm_adm2'

drivers_data = drivers_aligned.band_data
drivers_data.name = 'drivers'

tcd_thresholds_data = tcd_thresholds_aligned.band_data
tcd_thresholds_data.name = 'tcd_threshold'

In [7]:
emis_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_1/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2015_2016/_pixel_yr/4000_pixels/20250502/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_2015_2016__20250505_16_14_41.tif"])
node_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_1/land_state_node/standard_model/annual_intervals/2015_2016/4000_pixels/20250502/00N_020E__23_-4_24_-3__land_state_node_2015_2016.tif"])
print(emis_tile_uri)
print(node_tile_uri)

0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
dtype: object
0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
dtype: object


In [8]:
emis = xr.open_mfdataset(
    emis_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

nodes = xr.open_mfdataset(
    node_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

print(emis)
nodes

<xarray.Dataset>
Dimensions:      (x: 4000, y: 4000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -3.999 -3.999 -4.0 -4.0
    spatial_ref  int64 ...
Data variables:
    band_data    (y, x) float32 dask.array<chunksize=(400, 400), meta=np.ndarray>


<xarray.Dataset>
Dimensions:      (x: 4000, y: 4000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -3.999 -3.999 -4.0 -4.0
    spatial_ref  int64 ...
Data variables:
    band_data    (y, x) float64 dask.array<chunksize=(400, 400), meta=np.ndarray>

In [9]:
emis_sub, nodes_aligned = xr.align(emis, nodes, join="inner")
print(emis_sub)
nodes_aligned

<xarray.Dataset>
Dimensions:      (x: 4000, y: 4000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -3.999 -3.999 -4.0 -4.0
    spatial_ref  int64 ...
Data variables:
    band_data    (y, x) float32 dask.array<chunksize=(400, 400), meta=np.ndarray>


<xarray.Dataset>
Dimensions:      (x: 4000, y: 4000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -3.999 -3.999 -4.0 -4.0
    spatial_ref  int64 ...
Data variables:
    band_data    (y, x) float64 dask.array<chunksize=(400, 400), meta=np.ndarray>

In [10]:
node_data = nodes_aligned.band_data
node_data.name = 'state_node'
node_data

<xarray.DataArray 'state_node' (y: 4000, x: 4000)>
dask.array<getitem, shape=(4000, 4000), dtype=float64, chunksize=(400, 400), chunktype=numpy.ndarray>
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -3.999 -3.999 -4.0 -4.0
    spatial_ref  int64 ...
Attributes:
    AREA_OR_POINT:  Area

In [ ]:
tcl_years = range(1, 24)
drivers_cats = range(1, 6)
tcd_threshold_levels = range(1, 8)

In [ ]:
# gadm_adm0_ids = np.unique(gadm_adm0_data.data).compute()
# gadm_adm0_ids = gadm_adm0_ids[~np.isnan(gadm_adm0_ids)]

In [ ]:
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566., 570., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

In [ ]:
# gadm_adm1_ids = np.unique(gadm_adm1_data.data).compute()
# gadm_adm1_ids = gadm_adm1_ids[~np.isnan(gadm_adm1_ids)]

In [ ]:
gadm_adm1_ids = np.array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85], dtype=np.uint16)

In [ ]:
# gadm_adm2_ids = np.unique(gadm_adm2_data.data).compute()
# gadm_adm2_ids = gadm_adm2_ids[~np.isnan(gadm_adm2_ids)]

In [ ]:
gadm_adm2_ids = np.array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,
       182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194,
       195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207,
       208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220,
       221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233,
       234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246,
       247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259,
       260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272,
       273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285,
       286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298,
       299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311,
       312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324,
       325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337,
       338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350,
       351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363,
       364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376,
       377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389,
       390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402,
       403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415,
       416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428,
       429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441,
       442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454,
       455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467,
       468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480,
       481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493,
       494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506,
       507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519,
       520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532,
       533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545,
       546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558,
       559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571,
       572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584,
       585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597,
       598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610,
       611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623,
       624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
       637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649,
       650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662,
       663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675,
       676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688,
       689, 690, 691, 692, 693, 694, 695, 696, 697, 698, 699, 700, 701,
       702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714,
       715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727,
       728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740,
       741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753,
       754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766,
       767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779,
       780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792,
       793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805,
       806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818,
       819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830, 831,
       832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843, 844,
       845, 846, 847, 848, 849, 850, 851, 852, 853], dtype=np.uint16)

In [ ]:
from flox import ReindexArrayType, ReindexStrategy

tcl_by_year = xarray_reduce(
    areas_sub.band_data,
    tcl_data,
    drivers_data,
    tcd_thresholds_data,
    gadm_adm0_data,
    gadm_adm1_data,
    gadm_adm2_data,
    func='sum',
    expected_groups=(tcl_years, drivers_cats, tcd_threshold_levels, gadm_adm0_ids, gadm_adm1_ids, gadm_adm2_ids),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
    
)
tcl_by_year

In [11]:
node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
                       2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
                       2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
                      dtype=np.uint32)


In [12]:
from flox import ReindexArrayType, ReindexStrategy

emis_by_node = xarray_reduce(
    emis_sub.band_data,
    node_data,
    func='sum',
    expected_groups=(node_codes),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
    
)
emis_by_node

<xarray.DataArray 'band_data' (state_node: 25)>
dask.array<groupby_nansum, shape=(25,), dtype=float32, chunksize=(25,), chunktype=sparse.COO>
Coordinates:
    band         int64 1
    spatial_ref  int64 ...
  * state_node   (state_node) uint32 2211100 2211200 2212110 ... 5210000 5220000
Attributes:
    AREA_OR_POINT:  Area

In [14]:
result = emis_by_node.compute()

In [15]:
result

Format,coo
Data Type,float32
Shape,"(25,)"
nnz,25
Density,1.0
Read-only,True
Size,300
Storage ratio,3.00


In [16]:
sparse_data = result.data

# Step 3: Extract coordinates and values
dim_names = result.dims
indices = sparse_data.coords
values = sparse_data.data

# Step 4: Map dimension indices to coordinate values
coord_dict = {
    dim: result.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
coord_dict["value"] = values

df = pd.DataFrame(coord_dict)


In [18]:
df

,state_node,value
0,2211100,7.996428e+03
1,2211200,1.545713e+04
2,2212110,6.286086e+06
3,2212120,2.648156e+07
4,2212210,1.365178e+03
5,2212220,4.998512e+04
6,2214100,1.963823e+03
7,2214200,1.514823e+03
8,2215200,1.135479e+01
9,2221100,3.943771e+04


In [17]:
df.head()

,state_node,value
0,2211100,7.996428e+03
1,2211200,1.545713e+04
2,2212110,6.286086e+06
3,2212120,2.648156e+07
4,2212210,1.365178e+03


In [ ]:
df[(df.gadm_adm0 == 404) & (df.gadm_adm1 == 20) & (df.gadm_adm2 == 3) & (df.tcd_threshold == 5)]

In [ ]:
tcl_by_year.to_zarr("s3://gfw-data-lake/tsc_tree_cover_loss_drivers/v2023/raster/epsg-4326/zarr/gadm_adm2_results.zarr/", mode="w")